# BM25 Baseline Eval

The **eval** notebook: run it repeatedly as you tune. It loads the chunks
produced by `data_processing.ipynb`, builds the BM25 retriever, runs every
non-null query, and scores retrieval at the document level.

BM25 is the lexical baseline — pure keyword matching (TF saturation + IDF +
length normalization), no model, no embeddings. Everything fancier (dense,
hybrid, rerank) gets compared against these numbers.

**Prereqs:** `rag_eval/` uploaded next to this notebook, and
`data_processing.ipynb` already run (so `chunks/chunks.parquet` exists in S3).

## 1. Imports + session

`rank_bm25` is needed by the BM25 retriever — install it into the kernel if
the image doesn't already have it.

In [ ]:
import sys
!{sys.executable} -m pip install -q rank-bm25==0.2.2

sys.path.insert(0, ".")  # rag_eval/ next to this notebook
import io, os
import boto3
import pandas as pd

REGION = "us-east-1"
BUCKET = "rag-arch-eval-data-875699230934"
CHUNKS_KEY = "chunks/chunks.parquet"
DATA_PREFIX = "data"

s3 = boto3.client("s3", region_name=REGION)

def read_parquet_s3(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))

## 2. Load chunks + queries

Rebuild `Chunk` objects from the parquet (the retriever expects them), and
load the queries. We exclude `null_query` rows — they have no gold evidence,
so they're not part of retrieval scoring (they're for answer-level abstention
testing later).

In [ ]:
from rag_eval.chunking import Chunk
from rag_eval.data import load_queries  # reads data/queries.parquet locally...

chunks_df = read_parquet_s3(CHUNKS_KEY)
chunks = [Chunk(r.chunk_id, r.doc_id, r.text) for r in chunks_df.itertuples()]
print("chunks:", len(chunks))

# Load queries straight from S3 (don't rely on a local data/ dir in Studio).
q_df = read_parquet_s3(f"{DATA_PREFIX}/queries.parquet")

# Build (query, question_type, relevant_doc_ids) tuples; dedupe evidence titles;
# drop null queries (no evidence).
qrecs = []
for r in q_df.itertuples():
    rel = tuple(dict.fromkeys(e["title"] for e in r.evidence_list))
    if rel:  # excludes null_query
        qrecs.append({"query": r.query, "question_type": r.question_type, "rel": set(rel)})
print("non-null queries:", len(qrecs))

## 3. Build the BM25 index

`BM25Retriever.index()` tokenizes every chunk and builds the BM25Okapi index
in memory. For ~a few thousand chunks this is quick.

In [ ]:
import time
from rag_eval.retrievers import BM25Retriever, ranked_doc_ids

r = BM25Retriever(k1=1.5, b=0.75)
t0 = time.perf_counter()
r.index(chunks)
print(f"indexed {len(chunks)} chunks in {time.perf_counter()-t0:.1f}s")

## 4. Retrieve + score every query

For each query we retrieve chunks, collapse them to unique doc_ids
(`ranked_doc_ids`), and score against the gold evidence with the four
document-level metrics. We fetch extra chunks so the top-k *documents* fill
up even when several chunks of one article crowd the list.

In [ ]:
from rag_eval.metrics import METRICS

KS = [1, 3, 5, 10, 20]
FETCH = max(KS) * 5

rows = []
for q in qrecs:
    ranked = ranked_doc_ids(r.retrieve(q["query"], FETCH))
    row = {"question_type": q["question_type"], "n_relevant": len(q["rel"])}
    for k in KS:
        for name, fn in METRICS.items():
            row[f"{name}@{k}"] = fn(ranked, q["rel"], k)
    rows.append(row)

per_q = pd.DataFrame(rows)
print("scored", len(per_q), "queries")
per_q.head(3)

## 5. Overall + per-question-type results

The headline numbers. `recall@k` = fraction of gold docs found; `all_hit@k`
= did we get **every** gold doc (the strict multi-hop metric). The per-type
breakdown shows where BM25 struggles (usually the reworded inference queries,
where lexical overlap with the source is weakest).

In [ ]:
metric_cols = [c for c in per_q.columns if "@" in c]
print("=== OVERALL ===")
print(per_q[metric_cols].mean().round(4).to_string())

print("\n=== BY QUESTION TYPE (recall@10 / all_hit@10) ===")
by_type = per_q.groupby("question_type")[metric_cols].mean().round(4)
print(by_type[["recall@10", "all_hit@10", "mrr@10", "ndcg@10"]].to_string())

## 6. (optional) Save results to S3

Persist the per-query scores + a summary row, so results accumulate across
experiments the same way the local `results/runs.csv` does.

In [ ]:
RUN_NAME = "bm25_w256_o32_notebook"

summary = {"run": RUN_NAME, "n_queries": len(per_q), "n_chunks": len(chunks)}
summary.update(per_q[metric_cols].mean().round(4).to_dict())

for df, key in [
    (per_q, f"results/notebook/{RUN_NAME}_per_query.csv"),
    (pd.DataFrame([summary]), f"results/notebook/{RUN_NAME}_summary.csv"),
]:
    b = io.StringIO(); df.to_csv(b, index=False)
    s3.put_object(Bucket=BUCKET, Key=key, Body=b.getvalue().encode())
    print("wrote s3://%s/%s" % (BUCKET, key))